# Patrón Observer (Observador)

## 1) ¿Qué problema resuelve?
Cuando ocurre un cambio (evento) en un componente y **muchos otros componentes** deben reaccionar, pero **no queremos acoplar** al emisor con todos los receptores.

Ejemplo típico:
- Se crea una tarea → notificar por email + registrar auditoría + actualizar métricas.
- Si el “core” (TaskService) llama directamente a Email/Audit/Metrics, queda **rígido**: cada nuevo receptor obliga a modificar el core.

**Observer** permite:
- “Cuando pasa X, notifica a todos los interesados”
- Agregar o quitar reacciones **sin tocar** la lógica principal.

---

## 2) Idea central (one-to-many)
Un **Publisher/Subject** emite eventos.  
Muchos **Observers/Subscribers** se suscriben y reaccionan.

- **1 Publisher** → **N Subscribers**
- Comunicación basada en eventos, no en llamadas directas a servicios concretos.

---

## 3) Componentes del patrón (GoF)
### Subject / Publisher
- Mantiene una lista de observadores.
- Provee métodos típicos:
  - `attach/subscribe(observer)`
  - `detach/unsubscribe(observer)`
  - `notify(event, payload)`

### Observer / Subscriber
- Interfaz común: `update(...)` o `on_event(...)`
- Cada observer implementa su reacción ante eventos.

### ConcreteSubject / ConcreteObserver
- Implementaciones reales:
  - `Order` publica `ORDER_PAID`
  - `EmailNotifier` reacciona enviando correo
  - `AuditLogger` reacciona registrando auditoría

---

## 4) Flujo de ejecución
1. Observers se **suscriben** al Subject.
2. Ocurre un evento en el Subject (o en el servicio principal).
3. El Subject **notifica** a todos los observers.
4. Cada observer ejecuta su lógica.

En pseudoflujo:
- `bus.subscribe(EmailNotifier)`
- `bus.subscribe(AuditLogger)`
- `bus.publish("TASK_CREATED", {...})`
- Email/Audit/Metrics reaccionan automáticamente.

---

## 5) ¿Cuándo usarlo? (casos reales)
- **Notificaciones**: email, SMS, push, Slack.
- **Auditoría**: guardar logs de acciones importantes.
- **Métricas/telemetría**: conteos, timings, tracking.
- **Integraciones**: disparar procesos secundarios cuando ocurre un evento.
- **UI**: componentes reactivos al cambio de estado (muy común en frontend).

---

## 6) Ventajas
- **Bajo acoplamiento**: el publisher no conoce a los receptores concretos.
- **Extensibilidad**: agregas un nuevo observer sin modificar el core (principio OCP).
- **Separación de responsabilidades**: el core hace negocio; observers hacen side-effects.
- **Reusabilidad**: observers se pueden reutilizar en otros eventos.

---

## 7) Trade-offs y riesgos
### 7.1 Orden de ejecución
- ¿Importa si primero se registra auditoría y luego se envía email?
- En un bus simple, el orden es el de suscripción; en sistemas reales puede no estar garantizado.

### 7.2 Manejo de fallos
- Si un observer falla (ej. proveedor email cae), ¿bloquea a todos?
- Buen patrón: **aislar errores por observer** (try/except por handler).

### 7.3 “Event storms” (cascadas)
- Un observer puede publicar otro evento → se forman cascadas.
- Recomendación: limitar side-effects o diseñar contratos claros.

### 7.4 Suscripciones y fugas de memoria
- Si nunca desuscribes, en apps largas puedes acumular listeners.
- En sistemas distribuidos, esto se traduce a *consumer groups* y manejo de offsets.

### 7.5 Observabilidad (debug)
- A veces es difícil rastrear “quién reaccionó” a qué evento.
- Solución: logs estructurados y trazabilidad (event_id, correlation_id).

---

## 8) Observer “in-process” vs “distribuido”
### Observer in-process
- Todo ocurre en memoria, en la misma app.
- Ej: lista de callbacks.

### Observer distribuido (event-driven)
- Publicación en un broker (Kafka, RabbitMQ, SNS/SQS).
- Observers son consumidores.
- Implica: **entregas al menos una vez** (at-least-once), idempotencia, reintentos.

**Importante:** conceptualmente es el mismo patrón, solo cambia el medio.

---

## 9) Buenas prácticas
- Definir un **contrato de evento** estable:
  - `event_type` + `payload` con campos claros.
- Mantener observers enfocados en **side-effects**, no reglas core del negocio.
- Aislar fallos por observer.
- Si es distribuido: agregar `event_id`/`request_id` para **idempotencia**.
- Documentar eventos: “qué emite el sistema” y “quién los consume”.

---

## 10) Comparación rápida con patrones cercanos
- **Observer vs Mediator**:
  - Observer: broadcast (fan-out) a múltiples listeners.
  - Mediator: coordinación central para reducir acoplamiento entre módulos.
- **Observer vs Command**:
  - Observer: reacción a eventos.
  - Command: encapsular acciones como objetos (cola, retry, undo).

---

## 11) Mini-ejemplo conceptual
**Publisher:** `Button.click()`  
**Observers:** `BeepListener`, `ClickCounter`, `PrintLogger`

- Agregar `ConfettiListener` no requiere cambiar `Button`.
- Esa es la idea principal del patrón.